# LoRA-Spec Phase 1 Hypothesis Check

This notebook validates the core Phase 1 hypothesis on Google Colab Pro: a LoRA-adapted target model should reduce speculative decoding acceptance rates relative to the same target without the adapter.

## Runtime

Use `Runtime -> Change runtime type` and select:

- GPU: `A100`
- High-RAM if available

Colab Pro A100 has 40 GB VRAM, so this notebook uses `gpu_memory_utilization=0.85` to leave headroom for speculative decoding KV cache behavior.

In [ ]:
from pathlib import Path

repo_root = Path('/content/lora-spec')
if not repo_root.exists():
    raise RuntimeError('Clone or copy this repository to /content/lora-spec before running the notebook.')

%cd /content/lora-spec
!python -m pip install -U pip
!python -m pip install -e .[analysis,colab]


## Hugging Face access

The base Llama 3 models are gated on Hugging Face. Accept the model licenses on the model pages first, then authenticate in this notebook.

- `meta-llama/Meta-Llama-3-8B-Instruct`
- `meta-llama/Llama-3.2-1B-Instruct`

Public adapter examples used below:

- Medical: `bootscoder/Llama-3-Medical-8B-SFT-LoRA`
- Code: `AdnanRiaz107/CodeLLAMA3-8BI-APPS`
- Higher-rank extracted adapter: `grimjim/Llama-3-Instruct-Nephilim-v3-LoRA-8B`


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
from huggingface_hub import snapshot_download

TARGET_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
DRAFT_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTERS = {
    "medical": "bootscoder/Llama-3-Medical-8B-SFT-LoRA",
    "code": "AdnanRiaz107/CodeLLAMA3-8BI-APPS",
    "high_rank": "grimjim/Llama-3-Instruct-Nephilim-v3-LoRA-8B",
}

for repo_id in [TARGET_MODEL, DRAFT_MODEL, *ADAPTERS.values()]:
    snapshot_download(repo_id=repo_id, local_dir_use_symlinks=False)


## Run the hypothesis check

The runnable target below uses the code adapter because its checkpoint metadata names the same instruct target. Other adapters require their matching base model and must not be mixed into this run.

In [ ]:
!python scripts/validate_hypothesis.py \
  --target-model meta-llama/Meta-Llama-3-8B-Instruct \
  --draft-model meta-llama/Llama-3.2-1B-Instruct \
  --adapter-path AdnanRiaz107/CodeLLAMA3-8BI-APPS \
  --adapter-rank 16 \
  --adapter-domain code \
  --dataset tatsu-lab/alpaca \
  --num-prompts 24 \
  --speculation-length 4 \
  --gpu-memory-utilization 0.85 \
  --verbose


In [ ]:
import json
from pathlib import Path

result_files = sorted(Path("results/phase1").glob("phase1_validation_*.json"))
latest = result_files[-1]
payload = json.loads(latest.read_text())
payload.keys(), latest


In [ ]:
import matplotlib.pyplot as plt

baseline = payload["baseline"]
adapted = payload["adapted"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(["baseline", "adapted"], [baseline["acceptance_rate_overall"], adapted["acceptance_rate_overall"]])
axes[0].set_ylim(0, 1)
axes[0].set_title("Acceptance rate")

axes[1].bar(["baseline", "adapted"], [baseline["throughput_tps"], adapted["throughput_tps"]])
axes[1].set_title("Throughput (tokens/sec)")

plt.tight_layout()
plt.show()


## Interpretation

If the adapted run shows a clear drop in acceptance rate or throughput relative to baseline, that is a Phase 1 positive signal and the project should proceed to the larger characterization sweep.